# Point Transformer (v1), from scratch in PyTorch

Reimplementation of [Point Transformer](https://arxiv.org/abs/2012.09164) (Zhao et al., ICCV 2021) for **part segmentation** on ShapeNet-Part.

**The core idea.** Point Transformer replaces the fixed **max-pool** that earlier point networks (PointNet++) use to aggregate each local neighborhood with **vector self-attention**: each point's new feature is a *learned, content-and-position-dependent* weighted blend of its neighbors, with a separate weight per feature channel. It applies attention *locally* (over each point's k nearest neighbors) so it scales to large clouds, and it uses a learned relative-position encoding that the paper shows is essential.

This also brings in a full **U-Net** for dense per-point prediction: an encoder of residual transformer blocks with transition-down stages, and a decoder with transition-up (interpolation + skip connections) back to full resolution.

Trained on **ShapeNet-Part**: ~16,881 shapes, 16 object categories, 50 part labels total (each category has only 2-6 of them). Metric is **instance mIoU**.

> Runs on Colab, Kaggle, or locally (setup auto-detects). Enable a **GPU runtime**. Attention over kNN neighborhoods is heavier than PointNet++, so epochs are slower; on Kaggle use **Save Version** for background runs.

### 0. Setup

In [ ]:
# PyTorch Geometric is used ONLY for the dataset loader. The model is pure PyTorch.
!pip install -q torch-geometric

In [ ]:
# Portable setup: detect the platform and set data + checkpoint paths accordingly.
import os, sys

def detect_environment():
    if "COLAB_GPU" in os.environ or "google.colab" in sys.modules:
        return "colab"
    if os.path.exists("/kaggle"):
        return "kaggle"
    return "local"

env = detect_environment()

if env == "colab":
    from google.colab import drive
    try:
        drive.mount("/content/drive")                                  # persists data + checkpoints
        DATA_ROOT = "/content/drive/MyDrive/ptseg_data/ShapeNet"
        CKPT_DIR  = "/content/drive/MyDrive/ptseg_checkpoints"
    except Exception as e:
        print(f"Drive mount failed ({e}); falling back to Colab local storage (lost on disconnect).")
        DATA_ROOT, CKPT_DIR = "/content/ShapeNet", "/content/ptseg_checkpoints"
elif env == "kaggle":
    DATA_ROOT, CKPT_DIR = "/kaggle/working/ShapeNet", "/kaggle/working"
else:
    DATA_ROOT, CKPT_DIR = "./ShapeNet", "./checkpoints"

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Environment: {env}")
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"CKPT_DIR   : {CKPT_DIR}")

In [ ]:
import math, torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import ShapeNet
from torch.utils.data import DataLoader   # plain torch loader (PyG's ignores collate_fn)
import torch_geometric.transforms as T

# ---------------- config ----------------
NUM_POINTS  = 2048     # points per shape (standard for ShapeNet-Part)
NUM_PARTS   = 50       # total distinct part labels across all 16 categories
NUM_CATS    = 16       # object categories (fed as a one-hot hint, as in the paper)
K           = 16       # neighbors per attention / grouping (paper's best setting)
BATCH_SIZE  = 16
EPOCHS      = 100
LR          = 5e-4

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

## 1. Data: ShapeNet-Part

PyG downloads ShapeNet-Part with per-point **normals** (3 extra channels) and per-point part labels. Each point's input feature is its **normal (3) + a one-hot of its object category (16) = 19 channels** -- the category hint is standard for this benchmark and lets one model serve all 16 categories. We normalize each shape into a unit sphere.

The part labels are **global** (0-49 across all categories), but each category only uses a fixed subset of parts; we build that category-to-parts map so mIoU is computed correctly (a chair can't be scored on airplane parts).

> **Dataset download.** The cell below pulls ShapeNet-Part (with normals) from a Hugging Face mirror and lays it out where PyG expects, so the dataset loads without hitting Stanford's server (which is frequently down). First run only; it caches afterward.

In [ ]:
import os, zipfile, shutil, urllib.request

# PyG expects the extracted category folders directly under DATA_ROOT/raw/
raw_dir = os.path.join(DATA_ROOT, 'raw')
os.makedirs(raw_dir, exist_ok=True)
marker = os.path.join(raw_dir, '02691156')   # the airplane category folder = "already extracted"

if not os.path.isdir(marker):
    url = ("https://huggingface.co/datasets/wangps/shapenet_segmentation/"
           "resolve/main/shapenetcore_partanno_segmentation_benchmark_v0_normal.zip")
    zip_path = os.path.join(DATA_ROOT, 'shapenet.zip')
    print('downloading from HF mirror (~670 MB, a few minutes)...')
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(raw_dir)
    os.remove(zip_path)
    # the zip wraps everything in one folder; flatten it up into raw/
    inner = os.path.join(raw_dir, 'shapenetcore_partanno_segmentation_benchmark_v0_normal')
    if os.path.isdir(inner):
        for name in os.listdir(inner):
            shutil.move(os.path.join(inner, name), os.path.join(raw_dir, name))
        os.rmdir(inner)
    print('ready. raw/ now holds the category folders PyG expects.')
else:
    print('dataset already present, skipping download.')

In [ ]:
# category order matches PyG's internal label assignment
CATEGORIES = ['Airplane','Bag','Cap','Car','Chair','Earphone','Guitar','Knife','Lamp',
              'Laptop','Motorbike','Mug','Pistol','Rocket','Skateboard','Table']

pre_transform = T.NormalizeScale()                 # center + scale each shape to a unit sphere

train_set = ShapeNet(DATA_ROOT, categories=CATEGORIES, include_normals=True,
                     split='trainval', pre_transform=pre_transform)
test_set  = ShapeNet(DATA_ROOT, categories=CATEGORIES, include_normals=True,
                     split='test',     pre_transform=pre_transform)

# Build {category_index -> set of part labels it uses}, needed for correct mIoU.
# PyG stores the object category per shape in data.category.
from collections import defaultdict
cat_to_parts = defaultdict(set)
for d in train_set:
    cat_to_parts[int(d.category)].update(d.y.unique().tolist())
cat_to_parts = {c: sorted(p) for c, p in cat_to_parts.items()}
print('shapes:', len(train_set), 'train+val |', len(test_set), 'test')
print('example part map -> Airplane parts:', cat_to_parts[0], '| Chair parts:', cat_to_parts[4])

In [ ]:
def sample_points(data, n=NUM_POINTS):
    """Resample each shape to exactly n points (pad if fewer, random subsample if more),
    so we can batch them as dense (B, n, C) tensors."""
    N = data.pos.size(0)
    if N >= n:
        idx = torch.randperm(N)[:n]
    else:
        idx = torch.cat([torch.arange(N), torch.randint(0, N, (n - N,))])
    return data.pos[idx], data.x[idx], data.y[idx], int(data.category)

def collate(batch, train=False):
    """Turn a PyG batch into dense tensors in (B, N, C) form.
    Input feature = normal(3) + category one-hot(16) = 19 channels."""
    P, X, Y, C = [], [], [], []
    for data in batch:
        pos, feat, y, cat = sample_points(data)
        onehot = F.one_hot(torch.full((pos.size(0),), cat), NUM_CATS).float()
        X.append(torch.cat([feat, onehot], dim=-1))     # (N, 19)
        P.append(pos); Y.append(y); C.append(cat)
    p = torch.stack(P).to(device)                        # (B, N, 3)
    x = torch.stack(X).to(device)                        # (B, N, 19)
    y = torch.stack(Y).to(device)                        # (B, N)
    if train:                                            # light augmentation
        p = p + torch.randn_like(p) * 0.01
        p = p * torch.empty(p.size(0),1,1,device=device).uniform_(0.8,1.25)
    return p, x, y, torch.tensor(C)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          collate_fn=lambda b: collate(b, train=True))
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=lambda b: collate(b, train=False))

In [ ]:
# Sanity check the pipeline
p, x, y, c = next(iter(train_loader))
print('pos:', tuple(p.shape), '| feat:', tuple(x.shape), '| labels:', tuple(y.shape))
# expect (B,2048,3), (B,2048,19), (B,2048)

## 2. Geometry helpers

The same operations as PointNet++ -- pairwise distances, gather-by-index, farthest point sampling -- plus **kNN** (Point Transformer uses k nearest neighbors for its local attention, rather than ball query). `knn` caps k at the number of available points, which matters at the deepest stage where the cloud is down to a handful of points.

In [ ]:
def square_distance(src, dst):
    """Pairwise squared distances. src: (B,S,C), dst: (B,N,C) -> (B,S,N)."""
    B, S, _ = src.shape; N = dst.shape[1]
    d = -2 * torch.matmul(src, dst.permute(0, 2, 1))
    d += (src ** 2).sum(-1).view(B, S, 1)
    d += (dst ** 2).sum(-1).view(B, 1, N)
    return d

def index_points(points, idx):
    """Gather points by index. points: (B,N,C), idx: (B,S) or (B,S,K) -> (...,C)."""
    B = points.shape[0]
    vs = [B] + [1] * (idx.dim() - 1); rs = [1] + list(idx.shape[1:])
    b = torch.arange(B, device=points.device).view(vs).repeat(rs)
    return points[b, idx, :]

def farthest_point_sample(xyz, npoint):
    """Pick npoint well-spread centroids (each new pick is farthest from those chosen)."""
    B, N, _ = xyz.shape; dev = xyz.device
    c = torch.zeros(B, npoint, dtype=torch.long, device=dev)
    dist = torch.full((B, N), 1e10, device=dev)
    far = torch.randint(0, N, (B,), dtype=torch.long, device=dev)
    bi = torch.arange(B, device=dev)
    for i in range(npoint):
        c[:, i] = far
        cen = xyz[bi, far, :].view(B, 1, 3)
        d = ((xyz - cen) ** 2).sum(-1)
        dist = torch.minimum(dist, d)
        far = dist.argmax(1)
    return c

def knn(k, xyz, new_xyz):
    """k nearest neighbors of each point in new_xyz, among xyz. Returns indices (B,S,k).
    k is capped at the available point count (the deepest stage has very few points)."""
    k = min(k, xyz.shape[1])
    d = square_distance(new_xyz, xyz)
    return d.topk(k, dim=-1, largest=False)[1]

## 3. The Point Transformer layer

This is the one genuinely new module versus PointNet++ -- **vector self-attention** (Eq. 3 of the paper):

```
y_i = sum_j  softmax( gamma(q_i - k_j + delta) )  *  ( v_j + delta )
```

- `q, k, v` are linear projections of the features (query / key / value), exactly as in NLP attention.
- The relation is **q - k** (subtraction), not a dot product, so it stays a *vector* -> the MLP `gamma` turns it into a **per-channel weight vector** (this is *vector* attention; scalar attention would give one weight per neighbor).
- `delta` is a learned **position encoding** `theta(p_i - p_j)` from relative coordinates, added to *both* the attention weights and the values -- the paper's ablation shows this is worth ~6 mIoU.
- softmax is over the k neighbors; `*` is element-wise (per-channel modulation).

Compare to PointNet++, whose aggregation was just `max` over neighbors. Here the weights are learned and depend on content + geometry.

In [ ]:
class PointTransformerLayer(nn.Module):
    """Vector self-attention over each point's kNN neighborhood (paper Eq. 3 and 4)."""
    def __init__(self, dim, k=K):
        super().__init__()
        self.k = k
        self.q = nn.Linear(dim, dim)                # query
        self.k_ = nn.Linear(dim, dim)               # key
        self.v = nn.Linear(dim, dim)                # value
        self.pos = nn.Sequential(nn.Linear(3, dim), nn.ReLU(), nn.Linear(dim, dim))   # theta (delta)
        self.attn = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim)) # gamma

    def forward(self, x, p):                         # x:(B,N,dim)  p:(B,N,3)
        idx = knn(self.k, p, p)                      # (B,N,k) each point's neighbors
        q = self.q(x)                                # (B,N,dim)
        k = index_points(self.k_(x), idx)            # (B,N,k,dim)
        v = index_points(self.v(x), idx)             # (B,N,k,dim)
        rel = p.unsqueeze(2) - index_points(p, idx)  # (B,N,k,3) relative positions
        delta = self.pos(rel)                        # position encoding
        a = self.attn(q.unsqueeze(2) - k + delta)    # gamma(q - k + delta) -> (B,N,k,dim)
        a = torch.softmax(a, dim=2)                  # softmax over the k neighbors
        return torch.sum(a * (v + delta), dim=2)     # attn-weighted sum of (v + delta)

### The Point Transformer block

Wraps the attention layer in a **residual block** (linear -> point transformer layer -> linear, + skip connection), standard Transformer-block hygiene. The small helper applies BatchNorm over the flattened points.

In [ ]:
class PTBlock(nn.Module):
    """Residual block with the point transformer layer at its core."""
    def __init__(self, dim, k=K):
        super().__init__()
        self.l1 = nn.Linear(dim, dim); self.l2 = nn.Linear(dim, dim)
        self.ptl = PointTransformerLayer(dim, k)
        self.bn1 = nn.BatchNorm1d(dim); self.bn2 = nn.BatchNorm1d(dim)

    def _bn(self, bn, x):
        sh = x.shape
        return bn(x.reshape(-1, sh[-1])).reshape(*sh)

    def forward(self, x, p):
        r = x
        x = F.relu(self._bn(self.bn1, self.l1(x)))
        x = self.ptl(x, p)
        x = self._bn(self.bn2, self.l2(x))
        return F.relu(x + r)

## 4. Transition down and up (the U-Net glue)

- **Transition down** = the PointNet++ move: farthest-point-sample to a smaller set, kNN-group onto those centroids, linear + max-pool. Reduces point count, grows feature width and receptive field.
- **Transition up** = the segmentation decoder step: **trilinear (3-NN inverse-distance) interpolation** spreads the coarse features back onto the denser set, then a **skip connection** adds the matching encoder-stage features (summation here, as in the paper). This is the feature-propagation / U-Net idea the classification notebooks didn't need.

In [ ]:
class LBR(nn.Module):
    """Linear + BatchNorm + ReLU, applied per point."""
    def __init__(self, i, o):
        super().__init__(); self.lin = nn.Linear(i, o); self.bn = nn.BatchNorm1d(o); self.act = nn.ReLU()
    def forward(self, x):
        sh = x.shape; x = self.lin(x)
        x = self.bn(x.reshape(-1, x.shape[-1])).reshape(*sh[:-1], -1)
        return self.act(x)

class TransitionDown(nn.Module):
    """FPS to 1/ratio points, kNN-group, linear + max-pool. Shrinks the cloud."""
    def __init__(self, i, o, k=K, ratio=4):
        super().__init__(); self.k = k; self.ratio = ratio; self.mlp = LBR(i + 3, o)
    def forward(self, x, p):
        B, N, _ = p.shape
        n_new = max(1, N // self.ratio)
        fid = farthest_point_sample(p, n_new)
        new_p = index_points(p, fid)                          # (B, n_new, 3)
        idx = knn(self.k, p, new_p)                           # neighbors of each new point
        gx = index_points(x, idx)                             # (B, n_new, k, i)
        gp = index_points(p, idx) - new_p.unsqueeze(2)        # localized coords
        g = self.mlp(torch.cat([gx, gp], dim=-1))             # (B, n_new, k, o)
        return g.max(dim=2)[0], new_p                         # max-pool over neighbors

def three_interpolate(p_target, p_source, feat_source):
    """Give each target point a feature via inverse-distance weighting of its 3 nearest
    source points (trilinear interpolation)."""
    d = square_distance(p_target, p_source)
    K3 = min(3, p_source.shape[1])
    dist, idx = d.topk(K3, dim=-1, largest=False)
    w = 1.0 / (dist + 1e-8); w = w / w.sum(-1, keepdim=True)
    return (index_points(feat_source, idx) * w.unsqueeze(-1)).sum(dim=2)

class TransitionUp(nn.Module):
    """Interpolate coarse features onto the denser set, then add the skip-linked features."""
    def __init__(self, i, skip, o):
        super().__init__(); self.l1 = LBR(i, o); self.l2 = LBR(skip, o)
    def forward(self, x, p, skip_x, skip_p):
        up = three_interpolate(skip_p, p, self.l1(x))         # onto the denser (skip) set
        return up + self.l2(skip_x), skip_p

## 5. The segmentation network

A 5-stage U-Net: encoder downsamples with transition-downs (2048 -> 512 -> 128 -> 32 -> 8 points, widths 32 -> 512), decoder upsamples with transition-ups + skip connections back to all 2048 points, then a per-point head -> 50 part logits. `forward` takes positions `p` and features `x` and returns per-point logits.

In [ ]:
class PointTransformerSeg(nn.Module):
    def __init__(self, num_classes=NUM_PARTS, in_feat=19, k=K):
        super().__init__()
        self.inmlp = LBR(in_feat, 32); self.e1 = PTBlock(32, k)
        self.d1 = TransitionDown(32, 64, k);  self.e2 = PTBlock(64, k)
        self.d2 = TransitionDown(64, 128, k); self.e3 = PTBlock(128, k)
        self.d3 = TransitionDown(128, 256, k);self.e4 = PTBlock(256, k)
        self.d4 = TransitionDown(256, 512, k);self.e5 = PTBlock(512, k)
        self.u4 = TransitionUp(512, 256, 256); self.de4 = PTBlock(256, k)
        self.u3 = TransitionUp(256, 128, 128); self.de3 = PTBlock(128, k)
        self.u2 = TransitionUp(128, 64, 64);   self.de2 = PTBlock(64, k)
        self.u1 = TransitionUp(64, 32, 32);    self.de1 = PTBlock(32, k)
        self.head = nn.Sequential(LBR(32, 32), nn.Linear(32, num_classes))

    def forward(self, p, x):                     # p:(B,N,3)  x:(B,N,19)
        x0 = self.e1(self.inmlp(x), p); p0 = p   # stage 1  (N)
        x1, p1 = self.d1(x0, p0); x1 = self.e2(x1, p1)   # stage 2  (N/4)
        x2, p2 = self.d2(x1, p1); x2 = self.e3(x2, p2)   # stage 3  (N/16)
        x3, p3 = self.d3(x2, p2); x3 = self.e4(x3, p3)   # stage 4  (N/64)
        x4, p4 = self.d4(x3, p3); x4 = self.e5(x4, p4)   # stage 5  (N/256)
        y, _ = self.u4(x4, p4, x3, p3); y = self.de4(y, p3)   # decode + skip
        y, _ = self.u3(y,  p3, x2, p2); y = self.de3(y, p2)
        y, _ = self.u2(y,  p2, x1, p1); y = self.de2(y, p1)
        y, _ = self.u1(y,  p1, x0, p0); y = self.de1(y, p0)
        return self.head(y)                       # (B, N, num_classes)

## 6. Instance mIoU + training loop

**Instance mIoU** is the ShapeNet-Part metric: for each shape, compute IoU per part *within that shape's category* (so a chair is only scored on chair parts), average those, then average over all shapes. Masking logits to the category's valid parts is what makes the metric correct and also helps the prediction.

In [ ]:
@torch.no_grad()
def shape_miou(logits, y, cats):
    """Instance mIoU over a batch. logits:(B,N,50)  y:(B,N)  cats:(B,) category indices."""
    ious = []
    for b in range(logits.size(0)):
        parts = cat_to_parts[int(cats[b])]
        # restrict prediction to this category's valid parts
        masked = logits[b][:, parts]
        pred = torch.tensor(parts, device=logits.device)[masked.argmax(1)]
        per_part = []
        for part in parts:
            pr = pred == part; gt = y[b] == part
            inter = (pr & gt).sum().item(); union = (pr | gt).sum().item()
            per_part.append(1.0 if union == 0 else inter / union)   # absent part -> IoU 1
        ious.append(sum(per_part) / len(per_part))
    return ious

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss, all_ious, n = 0.0, [], 0
    for p, x, y, cats in loader:
        with torch.set_grad_enabled(train):
            logits = model(p, x)
            loss = F.cross_entropy(logits.reshape(-1, NUM_PARTS), y.reshape(-1))
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item() * p.size(0); n += p.size(0)
        all_ious += shape_miou(logits, y, cats)
    return total_loss / n, sum(all_ious) / len(all_ious)

def train(model, tag='pt'):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
    best, ckpt = 0.0, os.path.join(CKPT_DIR, f'pointtransformer_{tag}_best.pth')
    history = {'train_miou': [], 'test_miou': [], 'train_loss': []}
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_miou = run_epoch(model, train_loader, optimizer)
        _,       te_miou = run_epoch(model, test_loader)
        scheduler.step()
        for k_, v in zip(history, (tr_miou, te_miou, tr_loss)): history[k_].append(v)
        print(f'[{tag}] epoch {epoch:3d} | loss {tr_loss:.3f} | train mIoU {tr_miou:.3f} | test mIoU {te_miou:.3f}')
        if te_miou > best:
            best = te_miou
            torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                        'best_miou': best, 'history': history}, ckpt)
    print(f'[{tag}] done. best test mIoU: {best:.3f}')
    return model, history

### 7. Train

In [ ]:
model, hist = train(PointTransformerSeg(), 'pt')
# Expect instance mIoU in the ~0.83-0.85 range (paper reports 86.6% with the full recipe).

## 8. Visualizations

A mIoU/loss curve, a ground-truth-vs-prediction grid, and spinning GIFs of individual shapes colored by part (ground truth and prediction side by side). `pick_shape_seg` auto-selects a cleanly-segmented example; `upright` fixes ShapeNet's axis convention so shapes stand up rather than lying on their side.

In [ ]:
import matplotlib.pyplot as plt

def plot_curves(history, tag=''):
    ep = range(1, len(history['test_miou']) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(ep, history['train_miou'], label='train')
    ax[0].plot(ep, history['test_miou'],  label='test')
    ax[0].set(xlabel='epoch', ylabel='instance mIoU', title=f'mIoU {tag}'); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(ep, history['train_loss'], color='crimson')
    ax[1].set(xlabel='epoch', ylabel='loss', title=f'Train loss {tag}'); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

plot_curves(hist, 'Point Transformer')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

# high-contrast part palette (clearer than matplotlib's default tab10)
PART_COLORS = np.array([
    [0.90,0.10,0.10],[0.12,0.47,0.80],[0.17,0.80,0.75],[0.85,0.60,0.10],
    [0.80,0.25,0.75],[0.40,0.75,0.20],[0.55,0.35,0.20],[0.30,0.30,0.85]])

def upright(pos):
    """ShapeNet stores shapes y-up; matplotlib is z-up. Rotate so objects stand upright."""
    x, y, z = pos.T
    return np.stack([x, -z, y], axis=1)

@torch.no_grad()
def pick_shape_seg(model, dataset, want_cat, max_tries=400):
    """Find a shape of want_cat (category name) that the model segments cleanly."""
    model.eval()
    target = CATEGORIES.index(want_cat)
    loader = DataLoader(dataset, batch_size=1, shuffle=True,
                        collate_fn=lambda b: collate(b, train=False))
    best = None
    for n, (p, x, y, cats) in enumerate(loader):
        if n >= max_tries: break
        if int(cats[0]) != target: continue
        parts = cat_to_parts[int(cats[0])]
        pred = torch.tensor(parts)[model(p, x)[0][:, parts].argmax(1).cpu()]
        acc = (pred == y[0].cpu()).float().mean().item()      # pick the best-matching one
        if best is None or acc > best[0]:
            best = (acc, p[0].cpu().numpy(), pred.numpy(), y[0].cpu().numpy(), parts)
        if best[0] > 0.97: break
    print(f'picked {want_cat}: point-accuracy {best[0]:.3f}')
    return best

def seg_spin_compare(shape, filename='seg_compare.gif'):
    """Spin ground-truth (left) and prediction (right) together, colored by part."""
    acc, pos, pred, gt, parts = shape
    local = {p: i for i, p in enumerate(parts)}
    cg = PART_COLORS[[local[int(l)] % len(PART_COLORS) for l in gt]]
    cp = PART_COLORS[[local[int(l)] % len(PART_COLORS) for l in pred]]
    x, y, z = upright(pos).T
    fig = plt.figure(figsize=(9,5), facecolor='white')
    axg = fig.add_subplot(1,2,1, projection='3d'); axp = fig.add_subplot(1,2,2, projection='3d')
    for ax, c, t in [(axg,cg,'ground truth'), (axp,cp,'prediction')]:
        ax.scatter(x,y,z,c=c,s=10,edgecolors='none'); ax.set_axis_off()
        ax.set_box_aspect([1,1,1]); ax.set_title(t, fontsize=11)
    def rot(a):
        axg.view_init(elev=35, azim=a); axp.view_init(elev=35, azim=a); return fig,
    animation.FuncAnimation(fig, rot, frames=range(0,360,4), interval=50).save(
        filename, writer='pillow', fps=20)
    plt.close(); print('saved', filename)

@torch.no_grad()
def seg_grid_curated(model, dataset, want_cats, filename='segmentations.png'):
    """One clean example per named category, upright and from a flattering angle."""
    model.eval()
    targets = [CATEGORIES.index(c) for c in want_cats]
    picked = {}   # cat_index -> (pos, pred, gt, parts)
    loader = DataLoader(dataset, batch_size=1, shuffle=True,
                        collate_fn=lambda b: collate(b, train=False))
    for p, x, y, cats in loader:
        ci = int(cats[0])
        if ci in targets and ci not in picked:
            parts = cat_to_parts[ci]
            pred = torch.tensor(parts)[model(p, x)[0][:, parts].argmax(1).cpu()]
            acc = (pred == y[0].cpu()).float().mean().item()
            if acc > 0.90:   # only keep a cleanly-segmented one
                picked[ci] = (p[0].cpu().numpy(), pred.numpy(), y[0].cpu().numpy(), parts)
        if len(picked) == len(targets): break

    n = len(picked)
    fig = plt.figure(figsize=(6, n*2.4), facecolor='white')
    for row, ci in enumerate([c for c in targets if c in picked]):
        pos, pred, gt, parts = picked[ci]
        local = {pp: j for j, pp in enumerate(parts)}
        xx, yy, zz = upright(pos).T
        for col, lab, title in [(1, gt, 'ground truth'), (2, pred, 'prediction')]:
            c = PART_COLORS[[local[int(l)] % len(PART_COLORS) for l in lab]]
            ax = fig.add_subplot(n, 2, row*2 + col, projection='3d')
            ax.scatter(xx, yy, zz, c=c, s=4, edgecolors='none')
            ax.set_axis_off(); ax.set_box_aspect([1,1,1]); ax.view_init(elev=30, azim=45)
            if row == 0: ax.set_title(title, fontsize=10)
    plt.tight_layout(); plt.savefig(filename, dpi=110); plt.show()
    print('shown categories:', [CATEGORIES[c] for c in targets if c in picked])

# the categories that look great as point clouds ,  recognizable from any angle
seg_grid_curated(model, test_set, ['Chair','Table','Lamp','Car','Mug','Laptop'])

# a static grid for the README, plus spinning compare GIFs for a couple of clean shapes
# seg_grid(model, test_set, n=6)
for cat in ['Chair', 'Airplane']:
    seg_spin_compare(pick_shape_seg(model, test_set, cat), filename=f'seg_{cat.lower()}.gif')

## Resuming from a checkpoint

Fresh session: run setup + imports + the helper and model-definition cells, then load.

In [ ]:
ckpt = torch.load(os.path.join(CKPT_DIR, 'pointtransformer_pt_best.pth'), map_location=device)
model = PointTransformerSeg().to(device)
model.load_state_dict(ckpt['model_state'])
print(f"loaded epoch {ckpt['epoch']} | test mIoU {ckpt['best_miou']:.3f}")
plot_curves(ckpt['history'], 'Point Transformer (reloaded)')

---

*The architecture lineage, if you're curious:* PointNet learns one global feature, PointNet++ adds hierarchical local neighborhoods aggregated by max-pool, and Point Transformer keeps that sample-group-aggregate skeleton but swaps the max-pool for learned vector self-attention. I implemented the first two [here](https://github.com/dushyantDevl/pointnet-family-pytorch) (classification on ModelNet40); this repo is the attention-based successor, on the harder task of dense part segmentation.